# Conditional DDPM — Interactive Attribute Sampler

Pick the conditioning attributes with dropdowns, tune the sampling knobs, and
generate a cartoon face from a **trained checkpoint**. You can also draw a
**real training image at random**, auto-fill its true attributes into the
dropdowns, and generate the model's version of the *same* conditioning
side-by-side — a quick eyeball check that the classifier-free guidance is
actually steering generation.

> Put this notebook in `notebooks/` under the project root, or set `PROJECT_DIR` below.
> Set `CKPT_PATH` to a checkpoint
> saved by `train.py` and `DATA_ROOT` to the Cartoon Set folder.

## Legend — what every parameter means

**Conditioning attributes** (read from the checkpoint's `attrs` / `attribute_dims`)

| Control | Meaning |
|---|---|
| `<attr>` dropdown (e.g. `eye_color`, `hair_color`, `face_color`) | Categorical value in `[0, cardinality)`. For Cartoon Set these are **palette indices**, not named colours — value `3` just means "the 3rd colour in that attribute's palette". |
| `∅ null (uncond)` option | Sets that attribute to index = `cardinality`, the special **null token** the U-Net was trained with. Picking it tells the conditional branch to *ignore* that attribute (useful to see the effect of dropping one condition). Real training samples never use it. |

**The `<attr>` dropdowns — what each conditioned attribute is**

These are the attributes `train.py` conditions on by default (Cartoon Set has 18
attributes in total; only these three are fed to the model). Each value is a
**palette index**, not a named colour — Cartoon Set ships no human-readable
labels, so which index maps to which colour is found empirically (hit *🎲 Random
training sample* to see real examples). The cardinalities below are the Cartoon
Set defaults; the authoritative numbers are whatever the checkpoint reports
(`DIMS`, printed by the load cell).

| Attribute | What it controls | Cardinality | Valid values (+ null token) |
|---|---|---|---|
| `eye_color`  | Colour of the irises | 5  | `0–4`  (+ `5`  = `∅` null) |
| `hair_color` | Colour of the hair   | 10 | `0–9`  (+ `10` = `∅` null) |
| `face_color` | Skin / face colour   | 11 | `0–10` (+ `11` = `∅` null) |

> **Want the actual colour of each index** (e.g. `eye_color 3 = teal`)? Run the
> **Colour legend** cell below — it recovers a swatch + hex + nearest colour
> name for every value directly from your training images, since Google ships no
> names.

**Sampling parameters**

| Control | Meaning |
|---|---|
| `guidance w` | Classifier-free guidance weight. The guided noise is `ε̃ = (1+w)·ε_cond − w·ε_uncond`. **`w=0`** → the plain conditional model; **larger `w`** sharpens attribute adherence but pushes samples out-of-distribution. For these colour attributes ~**`w=1`** is the sweet spot; higher `w` oversaturates and *lowers* measured fidelity. |
| `sampler` | **DDIM** = fast, few steps, deterministic when `eta=0`. **DDPM** = full ancestral chain over all `timesteps` (stochastic, slower, reference quality). |
| `DDIM steps` | Number of denoising steps for DDIM. More = sharper but slower. `50` is a good default. **Ignored by DDPM**, which always runs the full `timesteps` chain. |
| `DDIM eta` | Interpolates DDIM↔DDPM stochasticity: `0` = deterministic DDIM, `1` = DDPM-like noise injection. Only used by DDIM. |
| `seed` | Seeds the initial Gaussian noise `x_T`. Same seed + same params ⇒ identical image. Change it to draw a *different* sample of the same attribute combination. |

**Fixed by the checkpoint** (shown for reference, not editable)

| Field | Meaning |
|---|---|
| `image_size` | Output resolution (e.g. 32×32). |
| `timesteps` | Length of the diffusion schedule used at training (e.g. 1000). |
| weights | Loaded from the **`ema`** state dict — EMA weights give cleaner samples than the raw ones. |

> **Speed note.** With `w>0` every step does *two* U-Net forwards (cond + uncond).
> On CPU prefer DDIM with ≤50 steps; DDPM (1000 steps) for a single image is fine
> but takes ~1–2 min on a laptop CPU, near-instant on the GTX 1080.

In [ ]:
# --- configuration -----------------------------------------------------------
PROJECT_DIR = ".."                      # project root containing src/cartoon_diffusion
CKPT_PATH   = "ckpt.pt"                 # a checkpoint saved by train.py  (e.g. run_YYYYmmdd_HHMMSS.pt)
DATA_ROOT   = "data/cartoonset10k"     # Cartoon Set folder (only needed for the "random training sample" button)
DATA_LIMIT  = None                      # e.g. 2000 to load faster; None = all images

# --- imports -----------------------------------------------------------------
import os, sys
sys.path.insert(0, os.path.join(PROJECT_DIR, "src"))

import torch
import matplotlib.pyplot as plt

from cartoon_diffusion.model import UNet
from cartoon_diffusion.diffusion import GaussianDiffusion
from cartoon_diffusion.data import denormalize            # CartoonSetDataset imported lazily below

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# --- load the trained checkpoint --------------------------------------------
assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found: {CKPT_PATH!r}\n"
    "Train one first, e.g.:\n"
    "  python train.py --root data/cartoonset10k --steps 40000 --batch 128 --ckpt ckpt.pt\n"
    "then set CKPT_PATH above to that file."
)

ck         = torch.load(CKPT_PATH, map_location=device)
ATTRS      = ck["attrs"]                 # e.g. ['eye_color', 'hair_color', 'face_color']
DIMS       = ck["attribute_dims"]        # e.g. [5, 10, 11]  (cardinalities)
IMAGE_SIZE = ck["image_size"]
TIMESTEPS  = ck["timesteps"]

model = UNet(DIMS, image_size=IMAGE_SIZE).to(device)
model.load_state_dict(ck["ema"])         # EMA weights -> cleaner samples
model.eval()
diff = GaussianDiffusion(timesteps=TIMESTEPS)

print(f"checkpoint step : {ck.get('step', '?')}")
print(f"attributes      : {list(zip(ATTRS, DIMS))}")
print(f"image_size      : {IMAGE_SIZE}   timesteps: {TIMESTEPS}")

In [ ]:
# --- (optional) load the training set for the "random training sample" button ---
# If the data folder isn't present, generation still works — you just won't be
# able to auto-fill / compare against a real image.
ds = None
try:
    from cartoon_diffusion.data import CartoonSetDataset
    ds = CartoonSetDataset(
        DATA_ROOT, image_size=IMAGE_SIZE,
        cond_attributes=tuple(ATTRS), limit=DATA_LIMIT,
    )
    # Sanity: the dataset's attribute order/cardinalities must match the checkpoint.
    assert ds.attribute_dims == list(DIMS), (ds.attribute_dims, DIMS)
    print(f"training set loaded: {len(ds)} images")
except Exception as e:
    print("training set NOT loaded (random-sample button disabled):", e)

## Colour legend (index → colour), recovered from the data

Cartoon Set stores colours only as palette **indices** — Google publishes no
names — so this cell reads the *actual* colour of every `(attribute, value)`
straight from your training images and prints a swatch + hex + nearest common
colour name. Method: average many training images per value, keep the pixels
whose colour varies **specifically** with that attribute (between-class
variance, so unrelated regions and random noise cancel out), and take the median
colour there. This localises even tiny regions like the irises.

Runs one streaming pass over `LEGEND_POOL` images (~1 min on a laptop CPU, needs
the dataset loaded). Bump `LEGEND_RES` if a small region looks off.

In [ ]:
import numpy as np
from PIL import Image

assert ds is not None, "Run the dataset cell first (needs DATA_ROOT)."

LEGEND_RES    = 160     # analysis resolution (bigger -> better for tiny regions like eyes, slower)
LEGEND_POOL   = 1200    # number of training images to average over
LEGEND_TOPPCT = 99.0    # keep this top-% of the variance map as "the region this attribute controls"
LEGEND_BLUR   = 5       # box-blur (px) on the variance map, favours compact regions

def _load_rgb(path, size, bg):
    img = Image.open(path).convert("RGBA")
    b = Image.new("RGBA", img.size, tuple(bg) + (255,))
    img = Image.alpha_composite(b, img).convert("RGB").resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.float32)

def _boxblur(a, k):
    pad = k // 2
    ap = np.pad(a, ((pad, pad), (pad, pad)), mode="edge")
    c = np.cumsum(np.cumsum(ap, 0), 1)
    c = np.pad(c, ((1, 0), (1, 0)), mode="constant")
    H, W = a.shape
    return (c[k:k+H, k:k+W] - c[:H, k:k+W] - c[k:k+H, :W] + c[:H, :W]) / (k * k)

# nearest common-colour name (rough, just for a human-readable label)
_NAMED = {
    "black":(20,20,20), "grey":(128,128,128), "silver":(200,200,200),
    "white":(245,245,245), "red":(200,40,40), "orange":(230,140,40),
    "brown":(120,70,40), "auburn":(140,70,45), "tan":(210,180,140),
    "beige":(225,205,170), "peach":(245,205,175), "gold":(220,180,60),
    "blonde":(230,205,120), "yellow":(235,220,70), "olive":(128,128,40),
    "green":(60,150,70), "teal":(40,140,140), "cyan":(90,200,220),
    "blue":(60,90,200), "navy":(30,40,110), "purple":(120,70,170), "pink":(230,150,180),
}
def _name(rgb):
    r, g, b = [float(x) for x in rgb]
    return min(_NAMED, key=lambda k: (r-_NAMED[k][0])**2 + (g-_NAMED[k][1])**2 + (b-_NAMED[k][2])**2)

# one streaming pass: per-(attr,value) mean image, without holding all images in RAM
rng = np.random.default_rng(0)
pool = rng.choice(len(ds), size=min(LEGEND_POOL, len(ds)), replace=False)
R = LEGEND_RES
sums   = [np.zeros((c, R, R, 3), np.float32) for c in DIMS]
counts = [np.zeros(c, np.int64) for c in DIMS]
for i in pool:
    arr = _load_rgb(ds.paths[int(i)], R, ds.bg_color)
    lab = ds.labels[int(i)]
    for ai in range(len(ATTRS)):
        v = int(lab[ai]); sums[ai][v] += arr; counts[ai][v] += 1

color_legend = {}   # attr -> [(value, rgb, name), ...]
for ai, (attr, card) in enumerate(zip(ATTRS, DIMS)):
    valid = counts[ai] >= 3
    means = sums[ai] / np.maximum(counts[ai], 1)[:, None, None, None]
    B = _boxblur(means[valid].var(0).mean(2), LEGEND_BLUR)   # between-class variance -> the region
    sel = B >= np.percentile(B, LEGEND_TOPPCT)
    rows = []
    for v in range(card):
        if valid[v]:
            rgb = np.median(means[v][sel], axis=0)
            rows.append((v, rgb, _name(np.round(rgb).astype(int))))
    color_legend[attr] = rows

# text table
for attr, rows in color_legend.items():
    print(attr + ":")
    for v, rgb, nm in rows:
        r, g, b = np.round(rgb).astype(int)
        print(f"  {v:>2} = {nm:<7} #{r:02x}{g:02x}{b:02x}")
    print()

# visual swatch grid
ncol = max(len(r) for r in color_legend.values())
fig, axes = plt.subplots(len(color_legend), ncol,
                         figsize=(1.15 * ncol, 1.4 * len(color_legend)), squeeze=False)
for i, (attr, rows) in enumerate(color_legend.items()):
    for j in range(ncol):
        ax = axes[i][j]; ax.set_xticks([]); ax.set_yticks([])
        for s in ax.spines.values(): s.set_visible(False)
        if j < len(rows):
            v, rgb, nm = rows[j]
            ax.imshow(np.ones((8, 8, 3)) * (np.round(rgb) / 255.0))
            ax.set_title(f"{v}: {nm}", fontsize=8)
        else:
            ax.imshow(np.ones((8, 8, 3)))
    axes[i][0].set_ylabel(attr, fontsize=10, rotation=90, labelpad=10)
plt.tight_layout(); plt.show()

In [ ]:
# --- core generation + display helpers --------------------------------------
# label_vec: list[int] of length len(ATTRS), each value in [0, card]
#            (== card selects that attribute's null / unconditional token).
# Returns an (H, W, 3) numpy image in [0, 1].
@torch.no_grad()
def generate(label_vec, guidance_weight=1.0, sampler="ddim", steps=50, eta=0.0, seed=0):
    labels = torch.tensor([label_vec], dtype=torch.long, device=device)  # (1, n_attr)
    torch.manual_seed(seed)                                              # reproducible x_T
    if sampler == "ddpm":
        imgs = diff.ddpm_sample(model, labels, image_size=IMAGE_SIZE,
                                guidance_weight=guidance_weight)
    else:
        imgs = diff.ddim_sample(model, labels, image_size=IMAGE_SIZE,
                                guidance_weight=guidance_weight, steps=steps, eta=eta)
    return denormalize(imgs[0]).cpu().permute(1, 2, 0).numpy()

def _label_caption(label_vec):
    parts = []
    for name, card, v in zip(ATTRS, DIMS, label_vec):
        parts.append(f"{name}={'∅' if v == card else v}")
    return "  ".join(parts)

def show(gen_img, real_img=None, caption=""):
    n = 2 if real_img is not None else 1
    fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 3.4))
    axes = [axes] if n == 1 else list(axes)
    if real_img is not None:
        axes[0].imshow(real_img); axes[0].set_title("Real (training)"); axes[0].axis("off")
        axes[1].imshow(gen_img);  axes[1].set_title("Generated");       axes[1].axis("off")
    else:
        axes[0].imshow(gen_img);  axes[0].set_title("Generated");       axes[0].axis("off")
    fig.suptitle(caption, fontsize=9, y=0.02)
    plt.tight_layout(); plt.show()

In [ ]:
# --- interactive UI (ipywidgets) --------------------------------------------
import ipywidgets as widgets
from IPython.display import display

# one dropdown per conditioned attribute; last option is the CFG null token
attr_dropdowns = []
for name, card in zip(ATTRS, DIMS):
    options = [(str(i), i) for i in range(card)] + [("∅ null (uncond)", card)]
    attr_dropdowns.append(widgets.Dropdown(
        options=options, value=0, description=name,
        style={"description_width": "90px"}, layout=widgets.Layout(width="230px")))

w_guid   = widgets.FloatSlider(value=1.0, min=0.0, max=8.0, step=0.5,
                               description="guidance w", continuous_update=False,
                               style={"description_width": "90px"}, layout=widgets.Layout(width="330px"))
w_sampler= widgets.Dropdown(options=[("DDIM (fast)", "ddim"), ("DDPM (full)", "ddpm")],
                            value="ddim", description="sampler",
                            style={"description_width": "90px"}, layout=widgets.Layout(width="230px"))
w_steps  = widgets.IntSlider(value=50, min=10, max=250, step=10, description="DDIM steps",
                             continuous_update=False,
                             style={"description_width": "90px"}, layout=widgets.Layout(width="330px"))
w_eta    = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.1, description="DDIM eta",
                               continuous_update=False,
                               style={"description_width": "90px"}, layout=widgets.Layout(width="330px"))
w_seed   = widgets.IntText(value=0, description="seed",
                           style={"description_width": "90px"}, layout=widgets.Layout(width="200px"))

btn_rand = widgets.Button(description="🎲 Random training sample",
                          layout=widgets.Layout(width="230px"),
                          disabled=(ds is None))
btn_gen  = widgets.Button(description="Generate", button_style="primary",
                          layout=widgets.Layout(width="130px"))
out = widgets.Output()

state = {"real_img": None, "real_labels": None}   # last drawn training image

def _current_labels():
    return [dd.value for dd in attr_dropdowns]

def _on_random(_):
    import random
    idx = random.randrange(len(ds))
    img, lab = ds[idx]
    lab = lab.tolist()
    for dd, v in zip(attr_dropdowns, lab):      # auto-fill dropdowns
        dd.value = v
    state["real_img"]    = denormalize(img).permute(1, 2, 0).numpy()
    state["real_labels"] = lab
    with out:
        out.clear_output(wait=True)
        print(f"loaded training image #{idx}")
        show(gen_img=state["real_img"], real_img=None,
             caption="Real (training)  |  " + _label_caption(lab))

def _on_generate(_):
    labels = _current_labels()
    with out:
        out.clear_output(wait=True)
        print("generating…")
        gen = generate(labels, guidance_weight=w_guid.value, sampler=w_sampler.value,
                       steps=w_steps.value, eta=w_eta.value, seed=w_seed.value)
        # show the real image alongside only if the dropdowns still match it
        real = state["real_img"] if labels == state["real_labels"] else None
        cap = _label_caption(labels) + \
              f"   |  w={w_guid.value}  {w_sampler.value}" + \
              (f"  steps={w_steps.value}  eta={w_eta.value}" if w_sampler.value == "ddim" else "") + \
              f"  seed={w_seed.value}"
        out.clear_output(wait=True)
        show(gen_img=gen, real_img=real, caption=cap)

btn_rand.on_click(_on_random)
btn_gen.on_click(_on_generate)

ui = widgets.VBox([
    widgets.HTML("<b>Attributes</b>"),
    widgets.HBox(attr_dropdowns, layout=widgets.Layout(flex_flow="row wrap")),
    widgets.HTML("<b>Sampling</b>"),
    widgets.HBox([w_sampler, w_guid]),
    widgets.HBox([w_steps, w_eta]),
    widgets.HBox([w_seed]),
    widgets.HBox([btn_rand, btn_gen]),
    out,
])
display(ui)

## No-widgets fallback

If `ipywidgets` isn't available (or you just want a scripted call), use
`generate(...)` + `show(...)` directly. `label_vec` is one integer per attribute
in `ATTRS` order; use the attribute's cardinality as the value to select the
null token.

In [ ]:
# Example: first value of every attribute, mild guidance, fast DDIM.
label_vec = [0] * len(ATTRS)                     # e.g. eye_color=0, hair_color=0, face_color=0
img = generate(label_vec, guidance_weight=1.0, sampler="ddim", steps=50, seed=0)
show(img, caption=_label_caption(label_vec) + "  |  w=1.0  ddim  steps=50")